# Firecrawl

Firecrawl can be used to search the web, scrape and crawl pages

## Firecrawl web search

In [ ]:
import requests
import os
from dotenv import load_dotenv

url = "https://api.firecrawl.dev/v2/search"

payload = {
    "query": "what are the latest status on Jensen Huang?",
    "sources": ["web"],
    "categories": [],
    "limit": 10,
    "scrapeOptions": {
        "onlyMainContent": True,
        "maxAge": 172800000,
        "parsers": ["pdf"],
        "formats": [],
    },
}

headers = {
    "Authorization": f"Bearer {os.environ['FIRECRAWL_API_KEY']}",
    "Content-Type": "application/json",
}

response = requests.post(url, json=payload, headers=headers)


In [24]:
result = response.json()
result.keys()

dict_keys(['success', 'data', 'creditsUsed', 'id'])

In [26]:
result["creditsUsed"]

2

In [28]:
result["data"].keys()

dict_keys(['web'])

In [31]:
len(result["data"]["web"])

10

In [ ]:
result["data"]["web"]

In [35]:

with open("jensen_huang.json", "w") as f:
    f.write(f"{result["data"]["web"]}")

## Crawl several pages

In [ ]:
from firecrawl import Firecrawl
from firecrawl.v2.types import ScrapeOptions

load_dotenv()

app = Firecrawl()

scrape_opts = ScrapeOptions(
    only_main_content=True,
    max_age=172800000,
    parsers=["pdf"],
    formats=["markdown"]
)

crawl_result = app.crawl(
    "https://www.myh.se/yrkeshogskolan",
    sitemap="include",
    crawl_entire_domain=False,
    limit=10,
    scrape_options=scrape_opts
)


In [42]:
crawl_result.__dict__.keys()

dict_keys(['id', 'status', 'total', 'completed', 'credits_used', 'expires_at', 'next', 'warning', 'data'])

In [43]:
crawl_result.credits_used

10

In [45]:
len(crawl_result.data)

10

In [52]:
crawl_result.data[0].__dict__.keys()

dict_keys(['agent_hints', 'markdown', 'html', 'raw_html', 'json', 'summary', 'metadata', 'links', 'images', 'screenshot', 'audio', 'video', 'actions', 'answer', 'highlights', 'warning', 'change_tracking', 'branding', 'product', 'menu', 'pages', 'blocks', 'tools'])

In [61]:
from pathlib import Path 

data_path = Path("myh_data")
data_path.mkdir(exist_ok=True)

for document in crawl_result.data:
    print(document.metadata.title)
    with open(data_path / f"{document.metadata.title}.md", "w") as f:
        f.write(document.markdown)

Yrkeshögskolan - Myndigheten för yrkeshögskolan
Ansök om att bedriva utbildning - Myndigheten för yrkeshögskolan
Anordnarkompetens - Myndigheten för yrkeshögskolan
Områdesanalyser och regional efterfrågan - Myndigheten för yrkeshögskolan
Ansökan program (YH-utbildningar från 100 poäng) - Myndigheten för yrkeshögskolan
Ansökningsomgången 2027 - Myndigheten för yrkeshögskolan
Ansökningsomgången 2026 - Myndigheten för yrkeshögskolan
Statsbidrag och schablonnivåer - Myndigheten för yrkeshögskolan
Nationellt likvärdigt innehåll (NLI) - Myndigheten för yrkeshögskolan
Smala yrkesområden - Myndigheten för yrkeshögskolan


## Scrape a single page

In [ ]:
from firecrawl import Firecrawl

app = Firecrawl()

data = app.scrape(
    "https://pydantic.dev/docs/ai/evals/evaluators/llm-judge/",
    only_main_content=True,
    max_age=172800000,
    parsers=["pdf"],
    formats=["markdown"],
)



In [65]:
data.__dict__.keys()

dict_keys(['agent_hints', 'markdown', 'html', 'raw_html', 'json', 'summary', 'metadata', 'links', 'images', 'screenshot', 'audio', 'video', 'actions', 'answer', 'highlights', 'warning', 'change_tracking', 'branding', 'product', 'menu', 'pages', 'blocks', 'tools'])

In [68]:
data.metadata.title

'LLM Judge Deep Dive | Pydantic Docs'

In [70]:
data.metadata.description

'Use an LLM as a judge in Pydantic Evals: grade outputs against a rubric with LLMJudge, get pass/fail or scores, pick the judge model and write good rubrics.'

In [72]:
data.metadata.__dict__.keys()

dict_keys(['title', 'description', 'url', 'language', 'keywords', 'robots', 'og_title', 'og_description', 'og_url', 'og_image', 'og_audio', 'og_determiner', 'og_locale', 'og_locale_alternate', 'og_site_name', 'og_video', 'favicon', 'dc_terms_created', 'dc_date_created', 'dc_date', 'dc_terms_type', 'dc_type', 'dc_terms_audience', 'dc_terms_subject', 'dc_subject', 'dc_description', 'dc_terms_keywords', 'modified_time', 'published_time', 'article_tag', 'article_section', 'source_url', 'status_code', 'scrape_id', 'num_pages', 'total_pages', 'content_type', 'proxy_used', 'timezone', 'cache_state', 'cached_at', 'credits_used', 'concurrency_limited', 'concurrency_queue_duration_ms', 'error'])

In [76]:
data.metadata.source_url

'https://pydantic.dev/docs/ai/evals/evaluators/llm-judge/'

In [78]:
data.metadata.proxy_used

'basic'

In [83]:
with open(f"{data.metadata.title}.md", "w") as f:
    f.write(data.markdown)